# Ropedia Academy — B4 · Implicit representations & SDF

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/B4.ipynb)

> **Trains an MLP to represent a signed-distance field and visualizes a 2D slice with the zero-level-set surface drawn as a contour.**
>
> 训练 MLP 来表示符号距离场，并可视化一个 2D 切片，把零等值面表面画成等高线。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and **visualizes the result with matplotlib** (the plot renders inline below the cell), so you learn the concept by executing and *seeing* it.

Colab's default runtime already includes `torch`, `numpy`, `networkx`, and `matplotlib`, so just press **Run all** — every cell goes green and a figure appears. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/B4

In [ ]:
import torch, torch.nn as nn, matplotlib.pyplot as plt

# Geometry as a FUNCTION f(x): signed distance; surface = {f = 0}.
sphere_sdf = lambda p, r=1.0: p.norm(dim=-1) - r
mlp = nn.Sequential(nn.Linear(3,64), nn.Softplus(), nn.Linear(64,64), nn.Softplus(), nn.Linear(64,1))
opt = torch.optim.Adam(mlp.parameters(), 1e-3)
for _ in range(600):                              # train an MLP to BE the SDF
    x = torch.randn(512, 3) * 1.5
    loss = ((mlp(x).squeeze(-1) - sphere_sdf(x))**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
print("MLP learned the SDF, residual:", round(loss.item(), 4))

# Visualize a 2D slice of the LEARNED field; the black contour is the surface f=0.
g = torch.stack(torch.meshgrid(torch.linspace(-1.6,1.6,90), torch.linspace(-1.6,1.6,90),
                               indexing='ij'), -1)
grid3 = torch.cat([g, torch.zeros(90,90,1)], -1).reshape(-1,3)
field = mlp(grid3).detach().reshape(90,90)
plt.figure(figsize=(4.5,4))
plt.contourf(field, levels=20, cmap='RdBu'); plt.colorbar()
plt.contour(field, levels=[0], colors='k', linewidths=2)
plt.title("learned SDF slice (black contour = surface, f=0)"); plt.show()

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/B4
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks